# Prepare QE Input From The Latest BCC Dataset Frame

This notebook does the three requested steps in one place:

1. Load the newest BCC `.npz` in `dataset/bcc/non-mag` and extract the last frame.
2. Either generate a zero-net noncollinear paramagnetic spin pattern in QE style, or import atom-resolved spin directions and restart-compatible magnitudes from a QE `.out` file. By default the notebook uses quasi-random spherical directions; paired-random is kept as an optional fallback mode.
3. Generate Maxwell-Boltzmann velocities and patch an `ATOMIC_VELOCITIES { a.u }` card into the QE input.

By default it reads the active external BCC dataset in `../dataset/bcc/non-mag` and writes the generated QE files into `../prepared_qe_inputs/latest_bcc`. Override the parameters below if you want a different dataset point or output location.

In [ ]:
from pathlib import Path
from pprint import pprint
import importlib
import sys

CWD = Path.cwd().resolve()
REPO_ROOT = None
for path in (CWD, *CWD.parents):
    direct_candidate = path
    nested_candidate = path / "IronCoreMD"
    if (direct_candidate / "codes" / "prepare_latest_bcc_qe_input.py").exists():
        REPO_ROOT = direct_candidate
        break
    if (nested_candidate / "codes" / "prepare_latest_bcc_qe_input.py").exists():
        REPO_ROOT = nested_candidate
        break
if REPO_ROOT is None:
    raise FileNotFoundError("Could not locate the IronCoreMD repo root from the current working directory.")
WORKSPACE_ROOT = REPO_ROOT.parent
CODES_DIR = REPO_ROOT / "codes"
if str(CODES_DIR) in sys.path:
    sys.path.remove(str(CODES_DIR))
sys.path.insert(0, str(CODES_DIR))

import prepare_latest_bcc_qe_input as latest_bcc_qe_input
latest_bcc_qe_input = importlib.reload(latest_bcc_qe_input)
infer_latest_bcc_npz = latest_bcc_qe_input.infer_latest_bcc_npz
parse_qe_atom_resolved_spin_guess = latest_bcc_qe_input.parse_qe_atom_resolved_spin_guess
prepare_bcc_qe_input = latest_bcc_qe_input.prepare_bcc_qe_input
build_collinear_random_test_input_from_frame = latest_bcc_qe_input.build_collinear_random_test_input_from_frame
SPIN_MODE_RANDOM_ZERO_NET = latest_bcc_qe_input.SPIN_MODE_RANDOM_ZERO_NET
SPIN_MODE_QUASI_RANDOM_ZERO_NET = latest_bcc_qe_input.SPIN_MODE_QUASI_RANDOM_ZERO_NET
MAGNETIC_MODE_NONCOLLINEAR_RANDOM = latest_bcc_qe_input.MAGNETIC_MODE_NONCOLLINEAR_RANDOM
MAGNETIC_MODE_COLLINEAR_RANDOM = latest_bcc_qe_input.MAGNETIC_MODE_COLLINEAR_RANDOM

DATASET_DIR = WORKSPACE_ROOT / "dataset" / "bcc" / "non-mag"
NPZ_PATH = None  # Example override: DATASET_DIR / "2.55_4000K.npz"
OUTPUT_DIR = WORKSPACE_ROOT / "prepared_qe_inputs" / "latest_bcc"

FRAME_INDEX = -1          # -1 means the last frame in the trajectory
TEMPERATURE_K = None      # None -> infer from the file name, e.g. 4000K
VELOCITY_SEED = 4000
RANDOM_SEED = 12345
MAGNETIC_MODE = MAGNETIC_MODE_COLLINEAR_RANDOM  # switch to MAGNETIC_MODE_NONCOLLINEAR_RANDOM to recover the original workflow
NONCOLLINEAR_SPIN_MODE = SPIN_MODE_QUASI_RANDOM_ZERO_NET  # used only when MAGNETIC_MODE is noncollinear_random
DEFAULT_QE_SPIN_OUTPUT_PATH = WORKSPACE_ROOT / "fe1.out"
QE_SPIN_OUTPUT_PATH = DEFAULT_QE_SPIN_OUTPUT_PATH if DEFAULT_QE_SPIN_OUTPUT_PATH.exists() else None
DEFAULT_QE_RESTART_SAVE_DIR = WORKSPACE_ROOT / "Fe_bcc_4x4x4_latest_noncollinear_paramagnetic.save"
QE_RESTART_SAVE_DIR = DEFAULT_QE_RESTART_SAVE_DIR if DEFAULT_QE_RESTART_SAVE_DIR.exists() else None
DEFAULT_QE_WFC_DIR = WORKSPACE_ROOT / "wavefunc"
QE_WFC_DIR = DEFAULT_QE_WFC_DIR if DEFAULT_QE_WFC_DIR.exists() else None
STARTING_MAGNETIZATION = 0.35  # used only for noncollinear mode
MIN_MOMENT = 0.3
MAX_MOMENT = 0.8

PSEUDO_DIR = WORKSPACE_ROOT
PSEUDO_FILE = "Fe.pbe-spn-kjpaw_psl.1.0.0.UPF"

DT_AU = 20.670
NSTEP = 400
ECUTWFC = 71.0
ECUTRHO = 496.0
DEGAUSS = 0.02
K_GRID = (4, 4, 4)

CONSTRAINED_MAGNETIZATION = True
LAMBDA_VALUE = 0.2
MIXING_BETA = 0.01
REMOVE_COM_DRIFT = True
RESCALE_EXACT = True
NOSYM = True

chosen_npz = NPZ_PATH if NPZ_PATH is not None else infer_latest_bcc_npz(DATASET_DIR)
effective_qe_spin_output_path = (
    QE_SPIN_OUTPUT_PATH if MAGNETIC_MODE == MAGNETIC_MODE_NONCOLLINEAR_RANDOM else None
)
print(f"Repo root:      {REPO_ROOT}")
print(f"Workspace:      {WORKSPACE_ROOT}")
print(f"Chosen NPZ:     {chosen_npz}")
print(f"Pseudo dir:     {PSEUDO_DIR}")
print(f"Output dir:     {OUTPUT_DIR.resolve()}")
print(f"Magnetic mode:  {MAGNETIC_MODE}")
print(f"Random seed:    {RANDOM_SEED}")
if MAGNETIC_MODE == MAGNETIC_MODE_COLLINEAR_RANDOM:
    print(f"Moment range:   [{MIN_MOMENT}, {MAX_MOMENT}]")
else:
    if effective_qe_spin_output_path is None:
        print(f"Spin mode:      {NONCOLLINEAR_SPIN_MODE}")
    else:
        qe_spin_output = Path(effective_qe_spin_output_path).resolve()
        parsed_spin_guess = parse_qe_atom_resolved_spin_guess(qe_spin_output)
        print("Spin source:    QE output")
        print(f"QE output:      {qe_spin_output}")
        pprint(parsed_spin_guess.summary_dict())

if QE_RESTART_SAVE_DIR is None:
    print("Restart seed:   none")
else:
    qe_restart_save_dir = Path(QE_RESTART_SAVE_DIR).resolve()
    restart_prefix = qe_restart_save_dir.name[:-5]
    restart_wfc_dir = Path(QE_WFC_DIR).resolve() if QE_WFC_DIR is not None else qe_restart_save_dir.parent
    restart_wfc_files = sorted(restart_wfc_dir.glob(f"{restart_prefix}.wfc*"))
    print(f"Restart save dir: {qe_restart_save_dir}")
    print(f"Restart WFC dir:  {restart_wfc_dir}")
    print(f"Restart WFC files found: {len(restart_wfc_files)}")

In [ ]:
example_qe_text, example_summary = build_collinear_random_test_input_from_frame(
    npz_path=chosen_npz,
    frame_index=FRAME_INDEX,
    natoms_preview=8,
    temperature_k=TEMPERATURE_K,
    random_seed=RANDOM_SEED,
    velocity_seed=VELOCITY_SEED,
    min_moment=MIN_MOMENT,
    max_moment=MAX_MOMENT,
    pseudo_dir=str(PSEUDO_DIR),
    pseudo_file=PSEUDO_FILE,
    dt_au=DT_AU,
    nstep=20,
    ecutwfc=ECUTWFC,
    ecutrho=ECUTRHO,
    degauss=DEGAUSS,
    k_grid=(1, 1, 1),
    mixing_beta=MIXING_BETA,
    nosym=NOSYM,
)

print("8-atom collinear random test input example (first 80 lines):\n")
print("\n".join(example_qe_text.splitlines()[:80]))
print("\n8-atom example magnetic summary:\n")
pprint(example_summary)

In [ ]:
result = prepare_bcc_qe_input(
    dataset_dir=DATASET_DIR,
    npz_path=chosen_npz,
    output_dir=OUTPUT_DIR,
    frame_index=FRAME_INDEX,
    temperature_k=TEMPERATURE_K,
    velocity_seed=VELOCITY_SEED,
    spin_seed=RANDOM_SEED,
    magnetic_mode=MAGNETIC_MODE,
    spin_mode=NONCOLLINEAR_SPIN_MODE,
    qe_spin_output_path=effective_qe_spin_output_path,
    qe_restart_save_dir=QE_RESTART_SAVE_DIR,
    qe_wfc_dir=QE_WFC_DIR,
    m_abs=STARTING_MAGNETIZATION,
    min_moment=MIN_MOMENT,
    max_moment=MAX_MOMENT,
    pseudo_dir=str(PSEUDO_DIR),
    pseudo_file=PSEUDO_FILE,
    dt_au=DT_AU,
    nstep=NSTEP,
    ecutwfc=ECUTWFC,
    ecutrho=ECUTRHO,
    degauss=DEGAUSS,
    k_grid=K_GRID,
    constrained_magnetization=CONSTRAINED_MAGNETIZATION,
    lambda_value=LAMBDA_VALUE,
    mixing_beta=MIXING_BETA,
    remove_com_drift=REMOVE_COM_DRIFT,
    rescale_exact=RESCALE_EXACT,
    nosym=NOSYM,
)

pprint(result.as_dict())

In [ ]:
final_input_path = Path(result.qe_input_final)
velocity_block_path = Path(result.velocity_block)
spin_parameters_path = Path(result.spin_parameters)

print(f"Magnetic mode used: {result.magnetic_mode}")
print(f"Spin source used: {result.spin_source}")
if result.qe_spin_output_path is not None:
    print(f"Imported QE spin output: {result.qe_spin_output_path}")
if result.qe_restart_save_dir is not None:
    print(f"Restart seed source: {result.qe_restart_save_dir}")
    if result.qe_wfc_dir is not None:
        print(f"Restart WFC dir: {result.qe_wfc_dir}")
    print(f"Staged restart save dir: {result.staged_restart_save_dir}")
    print(f"Density seeded: {result.restart_density_seeded}")
    print(f"Wavefunctions seeded: {result.restart_wfc_seeded}")
    print(f"Staged WFC files: {result.staged_wfc_count}")
if result.magnetic_mode == MAGNETIC_MODE_COLLINEAR_RANDOM:
    print(f"Number of positive moments: {result.positive_moment_count}")
    print(f"Number of negative moments: {result.negative_moment_count}")
    print(f"Average absolute magnetic moment: {result.mean_absolute_starting_magnetization:.6f}")
    print(f"Sum of initial magnetic moments: {result.sum_starting_magnetization:.6f}")
    print(f"Maximum magnetic moment: {result.max_starting_magnetization:.6f}")
    print(f"Minimum magnetic moment: {result.min_starting_magnetization:.6f}")
print(
    f"starting_magnetization range: {result.min_starting_magnetization:.6f} -> "
    f"{result.max_starting_magnetization:.6f} (mean {result.mean_starting_magnetization:.6f}, "
    f"mean abs {result.mean_absolute_starting_magnetization:.6f})"
)

print("Final QE input preview:\n")
print("\n".join(final_input_path.read_text().splitlines()[:60]))

print("\nVelocity card preview:\n")
print("\n".join(velocity_block_path.read_text().splitlines()[:8]))

print("\nSpin parameter preview:\n")
print("\n".join(spin_parameters_path.read_text().splitlines()[:12]))

In [ ]:
def extract_card_labels_from_file(path: Path, card_name: str, min_fields: int) -> list[str]:
    lines = path.read_text().splitlines()
    card_upper = card_name.upper()
    card_prefixes = (
        "&CONTROL",
        "&SYSTEM",
        "&ELECTRONS",
        "&IONS",
        "&CELL",
        "ATOMIC_SPECIES",
        "ATOMIC_POSITIONS",
        "ATOMIC_VELOCITIES",
        "K_POINTS",
        "CELL_PARAMETERS",
        "CONSTRAINTS",
        "OCCUPATIONS",
        "ATOMIC_FORCES",
        "SOLVENTS",
        "HUBBARD",
    )

    start = None
    for index, line in enumerate(lines):
        if line.strip().upper().startswith(card_upper):
            start = index
            break
    if start is None:
        raise ValueError(f"Could not find {card_name} in {path}")

    end = start + 1
    while end < len(lines):
        stripped = lines[end].strip().upper()
        if stripped.startswith(card_prefixes):
            break
        end += 1

    labels = []
    for line in lines[start + 1 : end]:
        stripped = line.strip()
        if not stripped or stripped.startswith(("!", "#")):
            continue
        fields = stripped.split()
        if len(fields) < min_fields:
            continue
        labels.append(fields[0])
    return labels


generated_input = Path(result.qe_input_final)
species_labels = extract_card_labels_from_file(generated_input, "ATOMIC_SPECIES", 3)
position_labels = extract_card_labels_from_file(generated_input, "ATOMIC_POSITIONS", 4)
velocity_labels = extract_card_labels_from_file(generated_input, "ATOMIC_VELOCITIES", 4)

line_by_line_match = position_labels == velocity_labels
if not line_by_line_match:
    mismatches = [
        (index + 1, pos_label, vel_label)
        for index, (pos_label, vel_label) in enumerate(zip(position_labels, velocity_labels))
        if pos_label != vel_label
    ]
    raise ValueError(f"Position/velocity label mismatch in generated QE input: {mismatches[:5]}")

species_set = set(species_labels)
if any(label not in species_set for label in position_labels):
    raise ValueError("Some ATOMIC_POSITIONS labels are missing from ATOMIC_SPECIES")
if any(label not in species_set for label in velocity_labels):
    raise ValueError("Some ATOMIC_VELOCITIES labels are missing from ATOMIC_SPECIES")

print(f"Generated input file: {generated_input}")
print(f"ATOMIC_SPECIES count: {len(species_labels)}")
print(f"ATOMIC_POSITIONS count: {len(position_labels)}")
print(f"ATOMIC_VELOCITIES count: {len(velocity_labels)}")
print(f"ATOMIC_POSITIONS and ATOMIC_VELOCITIES match line-by-line: {line_by_line_match}")
print(f"First 10 labels: {position_labels[:10]}")
print(f"Last 10 labels: {position_labels[-10:]}")
